# Vehicle Detection Fine-tuning — UVH-26 (Indian Traffic) on YOLOv8s

Fine-tunes YOLOv8s on the [UVH-26](https://huggingface.co/datasets/iisc-aim/UVH-26) dataset (Indian traffic CCTV footage, IISc Bengaluru) to detect 6 vehicle classes: `car`, `van`, `truck`, `bus`, `motorcycle`, `rickshaw`.

**Pipeline:** download subset → sanity check → COCO→YOLO conversion → fine-tune → evaluate → export.

Designed to run on **Kaggle Notebooks** (paths assume Kaggle's filesystem: `/kaggle/tmp`, `/kaggle/working`). See the repo README for setup notes, results, and known limitations.

> **Note on background execution:** an interactive Kaggle session dies if your browser disconnects or your PC sleeps. Once the early cells (download, sanity check) are verified, use **Save Version → Save & Run All (Commit)** to run training as a background batch job on Kaggle's servers.


In [ ]:
!pip install -q ultralytics huggingface_hub pycocotools

In [ ]:
import os

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF_TOKEN from Kaggle Secrets.")
except Exception:
    print("No HF_TOKEN found — downloading unauthenticated (slower, may hit rate limits).")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token

## 1. Download a subset of UVH-26

Images live under numbered shard folders in `data/` (e.g. `UVH-26-Train/data/000/`), **not** `images/` — confirmed by inspecting the actual repo file listing. Adjust `NUM_SHARDS_TRAIN` / `NUM_SHARDS_VAL` for more data. Downloads go to `/kaggle/tmp/` (scratch space — wiped on kernel restart, this is expected).

In [ ]:
from huggingface_hub import HfApi, snapshot_download

REPO_ID = "iisc-aim/UVH-26"
REVISION = "v1.0"
NUM_SHARDS_TRAIN = 2
NUM_SHARDS_VAL = 1

api = HfApi(token=hf_token)
all_files = api.list_repo_files(REPO_ID, repo_type="dataset", revision=REVISION)

def shard_folders(files, split_prefix):
    return sorted({
        f.split("/")[2] for f in files
        if f.startswith(f"{split_prefix}/data/") and len(f.split("/")) > 3
    })

train_shards = shard_folders(all_files, "UVH-26-Train")
val_shards = shard_folders(all_files, "UVH-26-Val")
print("Train shards available:", train_shards)
print("Val shards available:", val_shards)

chosen_train = train_shards[:NUM_SHARDS_TRAIN]
chosen_val = val_shards[:NUM_SHARDS_VAL]
print(f"\nUsing train shards: {chosen_train}, val shards: {chosen_val}")

allow_patterns = (
    [f"UVH-26-Train/data/{s}/*" for s in chosen_train]
    + [f"UVH-26-Val/data/{s}/*" for s in chosen_val]
    + ["UVH-26-Train/*.json", "UVH-26-Val/*.json", "LICENSE"]
)

local_dir = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    revision=REVISION,
    allow_patterns=allow_patterns,
    local_dir="/kaggle/tmp/uvh26_raw",
    token=hf_token,
    max_workers=4,
)
print("Downloaded to:", local_dir)

## 2. Inspect actual category names

In [ ]:
import json

TRAIN_JSON = "/kaggle/tmp/uvh26_raw/UVH-26-Train/UVH-26-MV-Train.json"
VAL_JSON = "/kaggle/tmp/uvh26_raw/UVH-26-Val/UVH-26-MV-Val.json"

with open(TRAIN_JSON) as f:
    train_coco = json.load(f)

print("Categories found in UVH-26-MV-Train.json:")
for c in train_coco["categories"]:
    print(f"  id={c['id']:<3} name={c['name']}")

## 3. Define class mapping

Category names come from the dataset's actual JSON (`Two-wheeler` / `Three-wheeler`, not the paper's release notes naming). Anything not listed is dropped (e.g. `Bicycle`, `Tempo-traveller`, `Others`).

In [ ]:
CLASS_MAP = {
    "Hatchback": "car",
    "Sedan": "car",
    "SUV": "car",
    "MUV": "car",
    "Van": "van",
    "LCV": "truck",
    "Truck": "truck",
    "Mini-bus": "bus",
    "Bus": "bus",
    "Two-wheeler": "motorcycle",
    "Three-wheeler": "rickshaw",
}

TARGET_CLASSES = ["car", "van", "truck", "bus", "motorcycle", "rickshaw"]
TARGET_CLASS_TO_ID = {name: i for i, name in enumerate(TARGET_CLASSES)}
print("Target classes (YOLO ids):", TARGET_CLASS_TO_ID)

## 4. Convert COCO annotations → YOLO format

Matches JSON entries to actual downloaded files by basename (robust to path differences), and reports how many images were kept vs. skipped and why.

In [ ]:
import os
from pathlib import Path

def build_file_index(images_root):
    index = {}
    for p in Path(images_root).rglob("*"):
        if p.is_file():
            index[p.name] = p
    return index

def coco_to_yolo(coco_json_path, images_root, out_images_dir, out_labels_dir):
    with open(coco_json_path) as f:
        coco = json.load(f)

    os.makedirs(out_images_dir, exist_ok=True)
    os.makedirs(out_labels_dir, exist_ok=True)

    cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
    img_id_to_info = {img["id"]: img for img in coco["images"]}
    file_index = build_file_index(images_root)
    print(f"Indexed {len(file_index)} actual image files under {images_root}")

    anns_by_image = {}
    for ann in coco["annotations"]:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    kept, skipped_no_labels, skipped_not_found = 0, 0, 0
    sample_missing = []

    for img_id, img_info in img_id_to_info.items():
        file_name = img_info["file_name"]
        basename = Path(file_name).name
        src_path = file_index.get(basename)
        if src_path is None:
            skipped_not_found += 1
            if len(sample_missing) < 3:
                sample_missing.append(file_name)
            continue

        w, h = img_info["width"], img_info["height"]
        lines = []
        for ann in anns_by_image.get(img_id, []):
            cat_name = cat_id_to_name.get(ann["category_id"])
            mapped = CLASS_MAP.get(cat_name)
            if mapped is None:
                continue
            cls_id = TARGET_CLASS_TO_ID[mapped]
            x, y, bw, bh = ann["bbox"]
            cx, cy = (x + bw / 2) / w, (y + bh / 2) / h
            nw, nh = bw / w, bh / h
            lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        if not lines:
            skipped_no_labels += 1
            continue

        dst_img = Path(out_images_dir) / basename
        if not dst_img.exists():
            os.symlink(src_path.resolve(), dst_img)

        label_path = Path(out_labels_dir) / (Path(basename).stem + ".txt")
        label_path.write_text("\n".join(lines))
        kept += 1

    print(
        f"{coco_json_path}: kept {kept}, "
        f"skipped_no_labels {skipped_no_labels}, skipped_not_found {skipped_not_found}"
    )
    if sample_missing:
        print("  Example missing file_name values:", sample_missing)
    return kept

BASE = "/kaggle/tmp/uvh26_yolo"
train_kept = coco_to_yolo(
    TRAIN_JSON,
    images_root="/kaggle/tmp/uvh26_raw/UVH-26-Train/data",
    out_images_dir=f"{BASE}/train/images",
    out_labels_dir=f"{BASE}/train/labels",
)
val_kept = coco_to_yolo(
    VAL_JSON,
    images_root="/kaggle/tmp/uvh26_raw/UVH-26-Val/data",
    out_images_dir=f"{BASE}/val/images",
    out_labels_dir=f"{BASE}/val/labels",
)

assert train_kept > 0, "No training images converted — check the diagnostics above before continuing."
assert val_kept > 0, "No validation images converted — check the diagnostics above before continuing."


## 5. Sanity check

Two quick checks before spending compute on training: class distribution (catches severe imbalance early) and a few images with boxes drawn (catches conversion bugs by eye).

In [ ]:
from collections import Counter

def class_distribution(labels_dir):
    counts = Counter()
    for label_file in Path(labels_dir).glob("*.txt"):
        for line in label_file.read_text().splitlines():
            cls_id = int(line.split()[0])
            counts[TARGET_CLASSES[cls_id]] += 1
    return counts

train_dist = class_distribution(f"{BASE}/train/labels")
print("Train class distribution (box counts):")
for cls in TARGET_CLASSES:
    print(f"  {cls}: {train_dist.get(cls, 0)}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

sample_images = random.sample(
    list(Path(f"{BASE}/train/images").glob("*")),
    min(4, len(list(Path(f"{BASE}/train/images").glob("*")))),
)

fig, axes = plt.subplots(1, len(sample_images), figsize=(5 * len(sample_images), 5))
if len(sample_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, sample_images):
    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)
    label_path = Path(f"{BASE}/train/labels") / (img_path.stem + ".txt")
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            cls_id, cx, cy, nw, nh = line.split()
            cls_id = int(cls_id)
            cx, cy, nw, nh = float(cx) * w, float(cy) * h, float(nw) * w, float(nh) * h
            x0, y0 = cx - nw / 2, cy - nh / 2
            rect = patches.Rectangle((x0, y0), nw, nh, linewidth=2, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.text(x0, y0 - 5, TARGET_CLASSES[cls_id], color="lime", fontsize=9, weight="bold")
    ax.set_title(img_path.name)
    ax.axis("off")

plt.tight_layout()
plt.show()
print("If boxes look correctly placed around vehicles, the conversion is good — proceed to training.")

## 6. Create `data.yaml`

In [ ]:
data_yaml = f"""train: {BASE}/train/images
val: {BASE}/val/images

nc: {len(TARGET_CLASSES)}
names: {TARGET_CLASSES}
"""

with open("/kaggle/tmp/data.yaml", "w") as f:
    f.write(data_yaml)

print(data_yaml)

## 7. Fine-tune YOLOv8s (with resume support)

Checkpoints save to `/kaggle/working/runs/` (persistent) every 5 epochs via `save_period=5`. The trainer auto-detects an existing `last.pt` to resume from if this cell is re-run after an interruption — a dropped connection costs a few epochs, not the whole run.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_DIR = "/kaggle/working/runs/detect"
RUN_NAME = "vehicle_finetune"
last_ckpt = Path(PROJECT_DIR) / RUN_NAME / "weights" / "last.pt"

if last_ckpt.exists():
    print(f"Found existing checkpoint at {last_ckpt} — resuming training.")
    model = YOLO(str(last_ckpt))
    results = model.train(resume=True)
else:
    print("No existing checkpoint — starting fresh from yolov8s.pt.")
    model = YOLO("yolov8s.pt")
    results = model.train(
        data="/kaggle/tmp/data.yaml",
        epochs=50,
        imgsz=640,
        batch=16,
        name=RUN_NAME,
        project=PROJECT_DIR,
        patience=10,
        save_period=5,
    )

## 8. Evaluate: precision / recall / mAP

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision (mean):", metrics.box.mp)
print("Recall (mean):", metrics.box.mr)

## 9. Confirm `best.pt` is ready

Sits in `/kaggle/working/runs/detect/vehicle_finetune/weights/best.pt` — downloadable from the notebook's Output panel once the run (or a Save & Run All commit) finishes.

In [ ]:
import os

best_pt_path = f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"
if os.path.exists(best_pt_path):
    size_mb = os.path.getsize(best_pt_path) / (1024 * 1024)
    print(f"Found best.pt ({size_mb:.1f} MB) at: {best_pt_path}")
    print("Download it from the Output panel on the right side of the Kaggle notebook.")
else:
    print("best.pt not found yet — make sure training (cell 9) finished successfully.")

---

## Results (reference run)

Trained 47/50 epochs (early stopping, patience=10) on 2 train shards / 1 val shard (~10k train images).

| Class | Precision | Recall | mAP50 | mAP50-95 |
|---|---|---|---|---|
| car | 0.863 | 0.912 | 0.943 | 0.840 |
| van | 0.586 | 0.528 | 0.546 | 0.489 |
| truck | 0.815 | 0.779 | 0.857 | 0.707 |
| bus | 0.832 | 0.777 | 0.861 | 0.732 |
| motorcycle | 0.884 | 0.831 | 0.921 | 0.709 |
| rickshaw | 0.900 | 0.873 | 0.934 | 0.800 |
| **all** | **0.813** | **0.783** | **0.844** | **0.713** |

**Known limitation:** `van` underperforms the other classes (mAP50 ~0.55 vs. 0.85+ elsewhere), directly traceable to low training example count (837 boxes vs. 3.8k–55k for other classes). Accepted tradeoff for now — more van examples would be the fix.

See the repo README for the full pipeline (tracking, counting, FastAPI backend, dashboard) this model feeds into.
